In [2]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator
import geopack.geopack as geopack
import joblib
import time
from datetime import datetime, timezone
from tqdm import tqdm

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
OUTPUT_FILENAME = "IGRF_Baked_Float32.pkl"
RE_M = 6.371e6 

# Simulation Time
# 12/31/2025 00:00:00 UTC
SIM_TIME = datetime(2025, 12, 31, 0, 0, 0, tzinfo=timezone.utc)

# Grid Settings (in Re)
GRID_MIN, GRID_MAX = -4.0, 4.0
RESOLUTION = 0.025 

# =============================================================================
# 2. HELPER: CALCULATE TRUE TILT (PSI)
# =============================================================================
def setup_geopack_environment(time_input):
    """Calculates the TRUE dipole tilt (PSI) without artificial clamping."""
    ut_timestamp = time_input.timestamp()
    geopack.recalc(ut_timestamp)
    psi = geopack.psi
    print(f"Geopack initialized for: {time_input}")
    print(f"True Dipole Tilt (PSI): {psi:.4f} rad ({np.degrees(psi):.2f} deg)")
    return psi

# =============================================================================
# 3. BAKING PROCESS
# =============================================================================
def bake_field():
    print("--- Starting IGRF Baking Process ---")
    start_t = time.time()

    # A. Setup Environment (True PSI)
    psi_rad = setup_geopack_environment(SIM_TIME)

    # B. Define Grid (In Re)
    axis_re = np.arange(GRID_MIN, GRID_MAX + RESOLUTION, RESOLUTION, dtype=np.float64)
    N = len(axis_re)
    print(f"Grid Definition: {GRID_MIN} to {GRID_MAX} Re (step {RESOLUTION})")
    print(f"Grid Shape: {N}x{N}x{N} = {N**3:,} points")

    # C. Pre-allocate Output Array 
    B_grid_T = np.zeros((N, N, N, 3), dtype=np.float32)

    # D. Loop and Calculate
    print("Calculating Field Values (IGRF ONLY)...")
    
    X, Y = np.meshgrid(axis_re, axis_re, indexing='ij')
    flat_X = X.ravel()
    flat_Y = Y.ravel()
    n_slice = len(flat_X)

    with tqdm(total=N, unit="slice", desc="Baking Slices") as pbar:
        for k, z_val in enumerate(axis_re):
            
            bx_slice = np.zeros(n_slice)
            by_slice = np.zeros(n_slice)
            bz_slice = np.zeros(n_slice)
            
            for i in range(n_slice):
                x, y = flat_X[i], flat_Y[i]
                
                # --- CRITICAL GEOMETRIC CHECK ---
                # Skip calculation if inside Earth (R < 1.0).
                # This avoids the singularity at R=0
                r_sq = x*x + y*y + z_val*z_val
                if r_sq < 1.0:
                    continue 

                # IGRF Internal ONLY
                bx_i, by_i, bz_i = geopack.igrf_gsm(x, y, z_val)
                
                bx_slice[i] = bx_i
                by_slice[i] = by_i
                bz_slice[i] = bz_i

            # Convert nT -> Tesla
            B_grid_T[:, :, k, 0] = bx_slice.reshape(N, N) * 1e-9
            B_grid_T[:, :, k, 1] = by_slice.reshape(N, N) * 1e-9
            B_grid_T[:, :, k, 2] = bz_slice.reshape(N, N) * 1e-9
            
            pbar.update(1)

    # =============================================================================
    # 3.5 PRECOMPUTE GRADIENTS & CURVATURE
    # =============================================================================
    print("\nPre-computing Gradients and Curvature vectors...")
    
    # 1. Spatial axes in meters for physical derivatives
    axis_meters = (axis_re * RE_M).astype(np.float32)
    
    # Extract components (already in Tesla)
    Bx = B_grid_T[:, :, :, 0]
    By = B_grid_T[:, :, :, 1]
    Bz = B_grid_T[:, :, :, 2]
    
    # 2. Calculate B magnitude safely
    Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)
    Bmag = np.maximum(Bmag, np.float32(1e-15)) # Prevent division by zero
    
    # Calculate unit vectors
    bx_hat = Bx / Bmag
    by_hat = By / Bmag
    bz_hat = Bz / Bmag
    
    # 3. Gradient of B magnitude (Resulting units: Tesla / meter)
    print("  >> Calculating grad(B)...")
    gradB_x, gradB_y, gradB_z = np.gradient(Bmag, axis_meters, axis_meters, axis_meters)
    
    gradB_x = gradB_x.astype(np.float32)
    gradB_y = gradB_y.astype(np.float32)
    gradB_z = gradB_z.astype(np.float32)
    
    # 4. Spatial derivatives of unit vectors
    print("  >> Calculating curvature components...")
    dbx_dx, dbx_dy, dbx_dz = np.gradient(bx_hat, axis_meters, axis_meters, axis_meters)
    dby_dx, dby_dy, dby_dz = np.gradient(by_hat, axis_meters, axis_meters, axis_meters)
    dbz_dx, dbz_dy, dbz_dz = np.gradient(bz_hat, axis_meters, axis_meters, axis_meters)
    
    # 5. Curvature vector: kappa = (b_hat dot nabla) b_hat
    kappa_x = (bx_hat * dbx_dx + by_hat * dbx_dy + bz_hat * dbx_dz).astype(np.float32)
    kappa_y = (bx_hat * dby_dx + by_hat * dby_dy + bz_hat * dby_dz).astype(np.float32)
    kappa_z = (bx_hat * dbz_dx + by_hat * dbz_dy + bz_hat * dbz_dz).astype(np.float32)

    # Clean up to free memory
    del dbx_dx, dbx_dy, dbx_dz, dby_dx, dby_dy, dby_dz, dbz_dx, dbz_dy, dbz_dz, bx_hat, by_hat, bz_hat

    # =============================================================================
    # 4. CREATE INTERPOLATORS & SAVE
    # =============================================================================
    print("\nCreating RegularGridInterpolators...")
    
    kwargs = {'bounds_error': False, 'fill_value': 0.0}
    
    # Original Fields
    interp_Bx = RegularGridInterpolator((axis_meters, axis_meters, axis_meters), Bx, **kwargs)
    interp_By = RegularGridInterpolator((axis_meters, axis_meters, axis_meters), By, **kwargs)
    interp_Bz = RegularGridInterpolator((axis_meters, axis_meters, axis_meters), Bz, **kwargs)
    
    # NEW: Gradient and Curvature Fields
    interp_gradBx = RegularGridInterpolator((axis_meters, axis_meters, axis_meters), gradB_x, **kwargs)
    interp_gradBy = RegularGridInterpolator((axis_meters, axis_meters, axis_meters), gradB_y, **kwargs)
    interp_gradBz = RegularGridInterpolator((axis_meters, axis_meters, axis_meters), gradB_z, **kwargs)
    
    interp_kappaX = RegularGridInterpolator((axis_meters, axis_meters, axis_meters), kappa_x, **kwargs)
    interp_kappaY = RegularGridInterpolator((axis_meters, axis_meters, axis_meters), kappa_y, **kwargs)
    interp_kappaZ = RegularGridInterpolator((axis_meters, axis_meters, axis_meters), kappa_z, **kwargs)

    output_data = {
        'Bx': interp_Bx, 'By': interp_By, 'Bz': interp_Bz,
        'gradBx': interp_gradBx, 'gradBy': interp_gradBy, 'gradBz': interp_gradBz,
        'kappaX': interp_kappaX, 'kappaY': interp_kappaY, 'kappaZ': interp_kappaZ,
        'description': 'Pure IGRF Grid (R>1.0). Input: Meters. Output: Tesla, T/m, 1/m.',
        'psi_used': psi_rad,
        'grid_bounds_re': [GRID_MIN, GRID_MAX]
    }

    print(f"Saving to {OUTPUT_FILENAME}...")
    joblib.dump(output_data, OUTPUT_FILENAME, compress=3)
    
    print(f"Done! Total time: {time.time() - start_t:.1f}s")

if __name__ == "__main__":
    bake_field()

--- Starting IGRF Baking Process ---
Geopack initialized for: 2025-12-31 00:00:00+00:00
True Dipole Tilt (PSI): -0.4432 rad (-25.39 deg)
Grid Definition: -4.0 to 4.0 Re (step 0.025)
Grid Shape: 321x321x321 = 33,076,161 points
Calculating Field Values (IGRF ONLY)...


Baking Slices:  44%|████▍     | 141/321 [14:19<19:23,  6.46s/slice]